In [ ]:
!pip install transformers sentencepiece sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
import sacrebleu
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ONE multilingual model
shared_model = M2M100ForConditionalGeneration.from_pretrained(
    "facebook/m2m100_418M"
).to(device)

shared_tokenizer = M2M100Tokenizer.from_pretrained(
    "facebook/m2m100_418M"
)

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#English->hindi
model_en_hi = shared_model
tok_en_hi = shared_tokenizer

#Hindi->English
model_hi_en = shared_model
tok_hi_en = shared_tokenizer

#English->Marathi
model_en_mr = shared_model
tok_en_mr = shared_tokenizer

#Marathi->English
model_mr_en = shared_model
tok_mr_en = shared_tokenizer

In [ ]:
def translate(text, model, tokenizer, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang

    encoded = tokenizer(text, return_tensors="pt").to(device)

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        num_beams=5,                 # improves quality
        length_penalty=1.0,
        early_stopping=True
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [ ]:
text = "The government has launched a new public health scheme."
translation = translate(text, model_en_mr, tok_en_mr, "en", "hi")

print("English:", text)
print("Hindi:", translation)

English: The government has launched a new public health scheme.
Hindi: सरकार ने एक नई सार्वजनिक स्वास्थ्य योजना शुरू की है।


In [ ]:
def multilingual_translate(text, src, tgt):

    if src == "en" and tgt == "hi":
        return translate(text, model_en_hi, tok_en_hi, "en", "hi")

    if src == "hi" and tgt == "en":
        return translate(text, model_hi_en, tok_hi_en, "hi", "en")

    if src == "en" and tgt == "mr":
        return translate(text, model_en_mr, tok_en_mr, "en", "mr")

    if src == "mr" and tgt == "en":
        return translate(text, model_mr_en, tok_mr_en, "mr", "en")

    if src == "hi" and tgt == "mr":
        return translate(text, model_hi_en, tok_hi_en, "hi", "mr")

    if src == "mr" and tgt == "hi":
        return translate(text, model_mr_en, tok_mr_en, "mr", "hi")

    return "Translation pair not supported."

In [ ]:
hindi_text = "सरकार ने एक नई सार्वजनिक स्वास्थ्य योजना शुरू की है।"
translation = translate(hindi_text, model_hi_en, tok_hi_en, "hi", "en")

print("Hindi:", hindi_text)
print("English:", translation)

Hindi: सरकार ने एक नई सार्वजनिक स्वास्थ्य योजना शुरू की है।
English: The government has launched a new public health plan.


In [ ]:
reference = ["The government has launched a new public health scheme."]
candidate = [translate("सरकार ने एक नई सार्वजनिक स्वास्थ्य योजना शुरू की है।", model_hi_en, tok_hi_en, "hi", "en")]

bleu = sacrebleu.corpus_bleu(candidate, [reference])

print("BLEU Score:", bleu.score)

BLEU Score: 78.25422900366438


In [ ]:
reference = ["The government has launched a new public health scheme."]
candidate = [translate("मला आनंद वाटतोय", model_mr_en, tok_mr_en, "mr", "en")]

bleu = sacrebleu.corpus_bleu(candidate, [reference])

print("BLEU Score:", bleu.score)

BLEU Score: 3.564186929405141


In [ ]:
while True:
    direction = input("Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): ")

    if direction == 'q':
        break

    text = input("Enter text: ")

    if direction == '1':
        print("Translation:", multilingual_translate(text, "en", "mr"))
    elif direction == '2':
        print("Translation:", multilingual_translate(text, "mr", "en"))
    elif direction == '3':
        print("Translation:", multilingual_translate(text, "en", "hi"))
    elif direction == '4':
        print("Translation:", multilingual_translate(text, "hi", "en"))
    elif direction == '5':
        print("Translation:", multilingual_translate(text, "mr", "hi"))
    elif direction == '6':
        print("Translation:", multilingual_translate(text, "hi", "mr"))
    elif direction == 'q':
        break
    else:
        print("Invalid option")

Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 1
Enter text: i am happy
Translation: मी खुश आहे
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 3
Enter text: i am happy
Translation: मैं खुश हूँ
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 1
Enter text: i am very happy
Translation: मी खूप खुश आहे
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 5
Enter text: मी खूप खुश आहे
Translation: मैं बहुत खुश हूँ
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 4
Enter text: मैं खुश हूँ
Translation: I am happy.
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 6
Enter text: मैं खुश हूँ
Translation: मी खुश आहे
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 2
Enter text: I am ill
Translation: I am ill
Enter direction (1: EN→MR, 2: MR→EN, 3: EN→HN, 4: HN→EN, 5: MR→HN, 6: HN→MR): 2
Enter text: 